# This Notebook is used to fine-tune Qwen3 with multilingual and monolingual data

## This notebook will create a lora-folder which contains the model and tokenizer that are needed in the QWEN3_prepare_evaluation notebook

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install transformers==4.51.3
    !pip install --no-deps unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset, concatenate_datasets
import pandas as pd
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
import gc

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-14B",
    max_seq_length = 2048,
    load_in_4bit = True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
dataset = load_dataset("textdetox/multilingual_paradetox")
dataset['en'] = dataset['en'].map(lambda x:{"lang": 'en'})
dataset['de'] = dataset['de'].map(lambda x:{"lang": 'de'})
dataset['ru'] = dataset['ru'].map(lambda x:{"lang": 'ru'})

ds = concatenate_datasets([dataset["en"], dataset["de"], dataset["ru"]], axis=0)


In [ ]:
system_prompt = """
You are an advanced language model tasked with detoxifying texts across multiple languages.
Your goal is to identify and neutralize toxic content while preserving the original meaning and language of the input text.

## Definition of Toxicity

Toxic language includes expressions that are offensive, hateful, abusive, discriminatory, or threatening.
It can be explicit, such as direct insults or slurs, or implicit, such as passive-aggressive comments or coded language.
Toxicity reduces the quality of conversation and harms communities.

## Steps to Success

1. Identify toxic content in the text while considering its context
2. Reformulate the text so it becomes a non-toxic text in the initial language
3. Check if it still contains toxic content
4. Make sure that the semantic content of the non-toxic text remains as similar as possible to the original text
5. Output only the detoxified final text

## Examples of Toxic Input and Non-Toxic Output Text Pairs in English

Toxic: "then all of a sudden i see her , shes now got the big phony tits and everything."
Non-toxic: "All of a sudden i see her, she is all grown up."

Toxic: "i would have lost my shit * years * ago."
Non-toxic: "I would have gone crazy years ago."

Toxic: "How about you fuck off and don't stalk my edits?"
Non-toxic: "How about you leave me alone and don't stalk my edits?"

Toxic: "You dumb motherfucker I am going to sue the shit out of you"
Non-toxic: "I am going to sue everything out of you"

Toxic: "таких тварей только могила исправит."
Non-toxic: "Этих людей ничего не исправит"

Toxic: "Wer die Grünen wählt ist entweder dumm oder ein Hurensohn #Schlafschafe"
Non-toxic: "Wer die Grünen wählt ist entweder uninformiert oder naiv"

## Task

Please detoxify the following toxic text using all the things provided above:
"""

In [ ]:
def generate_conversation(examples):
    toxic  = examples["toxic_sentence"]
    neutral = examples["neutral_sentence"]
    conversations = []
    for toxic, neutral in zip(toxic, neutral):
        conversations.append([
            {"role" : "user",      "content" : system_prompt + toxic},
            {"role" : "assistant", "content" : neutral},
        ])
    return { "conversations": conversations, }

In [ ]:
conversations = tokenizer.apply_chat_template(
    ds.map(generate_conversation, batched = True)["conversations"],
    tokenize = False,
)

In [ ]:
data = pd.concat([pd.Series(conversations),])
data.name = "text"
combined_dataset = Dataset.from_pandas(pd.DataFrame(data))
split = combined_dataset.shuffle(seed = 3407).train_test_split(test_size=0.25, seed=42)
combined_dataset = split["train"]
test_dataset = split["test"]

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = combined_dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 6,
        gradient_accumulation_steps = 2,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        report_to = "none",
    ),
)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 900 | Num Epochs = 3 | Total steps = 225
O^O/ \_/ \    Batch size per device = 6 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (6 x 2 x 1) = 12
 "-____-"     Trainable parameters = 128,450,560/14,000,000,000 (0.92% trained)


Step,Training Loss
1,2.403100
2,2.382400
3,2.413200
4,2.343400
5,2.148400
6,1.931400
7,1.697200
8,1.536800
9,1.370900
10,1.235300


In [ ]:
model.save_pretrained("lora")
tokenizer.save_pretrained("lora")

('lora/tokenizer_config.json',
 'lora/special_tokens_map.json',
 'lora/vocab.json',
 'lora/merges.txt',
 'lora/added_tokens.json',
 'lora/tokenizer.json')

In [ ]:
!zip -r lora_multi_e3_225.zip lora

  adding: lora/ (stored 0%)
  adding: lora/tokenizer.json (deflated 81%)
  adding: lora/adapter_config.json (deflated 57%)
  adding: lora/README.md (deflated 66%)
  adding: lora/special_tokens_map.json (deflated 69%)
  adding: lora/tokenizer_config.json (deflated 84%)
  adding: lora/added_tokens.json (deflated 68%)
  adding: lora/adapter_model.safetensors (deflated 8%)
  adding: lora/vocab.json (deflated 61%)
  adding: lora/merges.txt (deflated 57%)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp /content/lora_multi_e3_225.zip /content/drive/MyDrive/